# Neural Identifier Training with Particle Filters - Differential Drive Mobile Robot

In [28]:
import numpy as np
import plotly.graph_objects as go

In [29]:
# ============================================================
# PSO (Particle Swarm Optimization) for EKF Parameter Tuning
# ============================================================
class PSO_EKF_Optimizer:
    """
    Particle Swarm Optimization for tuning EKF-RHONN parameters.
    Optimizes: Q_init, R_init, P_init, eta
    """
    def __init__(self, n_particles=20, n_dimensions=4, bounds=None, 
                 w=0.9, c1=2.0, c2=2.0, max_iterations=50):
        """
        Initialize PSO optimizer for EKF parameters.
        
        Args:
            n_particles: Number of particles in swarm
            n_dimensions: Number of parameters to optimize (Q_init, R_init, P_init, eta)
            bounds: List of (min, max) tuples for each parameter
            w: Inertia weight
            c1: Cognitive parameter 
            c2: Social parameter
            max_iterations: Maximum number of iterations
        """
        self.n_particles = n_particles
        self.n_dimensions = n_dimensions
        self.max_iterations = max_iterations
        self.w = w
        self.c1 = c1
        self.c2 = c2
        
        # Default bounds for EKF parameters: [Q_init, R_init, P_init, eta]
        if bounds is None:
            self.bounds = [
                (1e-6, 1e-2),   # Q_init: process noise covariance
                (1e-4, 1e-1),   # R_init: measurement noise covariance  
                (0.1, 10.0),    # P_init: initial covariance
                (0.1, 2.0)      # eta: learning rate
            ]
        else:
            self.bounds = bounds
            
        # Initialize particles
        self.particles = np.zeros((n_particles, n_dimensions))
        self.velocities = np.zeros((n_particles, n_dimensions))
        self.personal_best_positions = np.zeros((n_particles, n_dimensions))
        self.personal_best_scores = np.full(n_particles, np.inf)
        self.global_best_position = np.zeros(n_dimensions)
        self.global_best_score = np.inf
        
        # Initialize particle positions randomly within bounds
        for i in range(n_particles):
            for j in range(n_dimensions):
                self.particles[i, j] = np.random.uniform(
                    self.bounds[j][0], self.bounds[j][1]
                )
        
        # Initialize personal best positions
        self.personal_best_positions = self.particles.copy()
        
        # Initialize velocities
        for j in range(n_dimensions):
            v_max = (self.bounds[j][1] - self.bounds[j][0]) * 0.1
            self.velocities[:, j] = np.random.uniform(-v_max, v_max, n_particles)
    
    def evaluate_particle(self, params, simulation_data):
        """
        Evaluate a single particle (set of EKF parameters).
        
        Args:
            params: [Q_init, R_init, P_init, eta]
            simulation_data: Dict containing simulation setup data
            
        Returns:
            fitness: Lower is better (MSE-based fitness)
        """
        try:
            Q_init, R_init, P_init, eta = params
            
            # Extract simulation data
            x_true = simulation_data['x_true']
            u_history = simulation_data['u_history'] 
            n_steps = simulation_data['n_steps']
            num_neurons = simulation_data['num_neurons']
            num_features = simulation_data['num_features']
            initial_weights = simulation_data['initial_weights']
            
            # Create EKF trainer with current parameters
            ekf_trainer = EKF_RHONN_Trainer(
                num_neurons, num_features,
                initial_weights=initial_weights,
                Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta
            )
            
            # Initialize estimation array
            x_hat_ekf = np.zeros((n_steps, 3))
            x_hat_ekf[0] = x_true[0]
            
            # Run simulation with current EKF parameters
            for k in range(n_steps - 1):
                u_current = u_history[k]
                
                # EKF update and prediction
                ekf_trainer.update(
                    chi_kp1=x_true[k+1], 
                    chi_k=x_hat_ekf[k], 
                    x_hat_previous=x_hat_ekf[k], 
                    u_input=u_current
                )
                
                # Predict next state
                x_state_for_z_ekf = np.copy(x_hat_ekf[k])
                x_state_for_z_ekf[0] = x_hat_ekf[k][0]
                
                x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)
                x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)  
                x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2], u_current)
            
            # Calculate fitness (total MSE - lower is better)
            mse_x = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
            mse_y = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
            mse_theta = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)
            
            # Weighted MSE (give more importance to position than orientation)
            fitness = 0.4 * mse_x + 0.4 * mse_y + 0.2 * mse_theta
            
            # Add penalty for extreme parameters to encourage stability
            penalty = 0
            if Q_init > 1e-3:  # Too high process noise
                penalty += (Q_init - 1e-3) * 100
            if R_init < 1e-3:  # Too low measurement noise
                penalty += (1e-3 - R_init) * 100
            if eta > 1.5:      # Too high learning rate
                penalty += (eta - 1.5) * 10
                
            return fitness + penalty
            
        except Exception as e:
            # Return very high fitness for failed evaluations
            print(f"Evaluation failed with params {params}: {e}")
            return 1e6
    
    def update_particles(self):
        """Update particle velocities and positions."""
        for i in range(self.n_particles):
            # Random factors
            r1, r2 = np.random.rand(2)
            
            # Update velocity
            inertia = self.w * self.velocities[i]
            cognitive = self.c1 * r1 * (self.personal_best_positions[i] - self.particles[i])
            social = self.c2 * r2 * (self.global_best_position - self.particles[i])
            
            self.velocities[i] = inertia + cognitive + social
            
            # Update position
            self.particles[i] += self.velocities[i]
            
            # Apply bounds
            for j in range(self.n_dimensions):
                if self.particles[i, j] < self.bounds[j][0]:
                    self.particles[i, j] = self.bounds[j][0]
                    self.velocities[i, j] *= -0.5  # Bounce back
                elif self.particles[i, j] > self.bounds[j][1]:
                    self.particles[i, j] = self.bounds[j][1]
                    self.velocities[i, j] *= -0.5  # Bounce back
    
    def optimize(self, simulation_data, verbose=True):
        """
        Run PSO optimization to find best EKF parameters.
        
        Args:
            simulation_data: Dict containing simulation setup
            verbose: Whether to print progress
            
        Returns:
            best_params: Optimized parameters [Q_init, R_init, P_init, eta]
            best_fitness: Best fitness achieved
            fitness_history: List of best fitness per iteration
        """
        fitness_history = []
        
        if verbose:
            print("🔍 Starting PSO optimization for EKF parameters...")
            print(f"Swarm size: {self.n_particles}, Max iterations: {self.max_iterations}")
        
        for iteration in range(self.max_iterations):
            # Evaluate all particles
            for i in range(self.n_particles):
                fitness = self.evaluate_particle(self.particles[i], simulation_data)
                
                # Update personal best
                if fitness < self.personal_best_scores[i]:
                    self.personal_best_scores[i] = fitness
                    self.personal_best_positions[i] = self.particles[i].copy()
                
                # Update global best
                if fitness < self.global_best_score:
                    self.global_best_score = fitness
                    self.global_best_position = self.particles[i].copy()
            
            fitness_history.append(self.global_best_score)
            
            if verbose and (iteration % 10 == 0 or iteration == self.max_iterations - 1):
                params = self.global_best_position
                print(f"Iteration {iteration+1:2d}: Best fitness = {self.global_best_score:.6f}")
                print(f"  Q_init={params[0]:.2e}, R_init={params[1]:.2e}, P_init={params[2]:.3f}, eta={params[3]:.3f}")
            
            # Update particles for next iteration
            if iteration < self.max_iterations - 1:
                self.update_particles()
        
        if verbose:
            print("✅ PSO optimization completed!")
            
        return self.global_best_position, self.global_best_score, fitness_history

In [30]:
def run_pso_ekf_optimization(n_steps=500, trajectory_type='figure8', 
                             pso_particles=20, pso_iterations=30, verbose=True):
    """
    Run PSO optimization for EKF parameters using a shorter simulation.
    
    Args:
        n_steps: Number of simulation steps for optimization
        trajectory_type: Type of robot trajectory 
        pso_particles: Number of PSO particles
        pso_iterations: Number of PSO iterations
        verbose: Print optimization progress
        
    Returns:
        optimized_params: Best EKF parameters found
        optimization_results: Dictionary with results and history
    """
    if verbose:
        print("🚀 Preparing simulation data for PSO optimization...")
    
    # Simulation settings (shorter for optimization)
    dt = 0.02
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)
    
    # Generate true system trajectory
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]
    u_history = []
    
    # Generate control inputs and true trajectory
    for k in range(n_steps - 1):
        t_current = k * dt
        u_current = generate_realistic_trajectory(t_current, trajectory_type)
        u_history.append(u_current)
        
        x_true[k+1] = plant(x_true[k], u_current, dt, 
                           process_noise_type='mixed', 
                           process_noise_std=0.0,  # Reduced noise for optimization
                           terrain_roughness=0.0, 
                           sensor_bias=[0.0, 0.0, 0.0])
    
    # RHONN configuration
    num_neurons = 3
    num_features = 17
    
    # Common initial weights for fair comparison
    np.random.seed(42)  # Reproducible results
    common_initial_weights = [
        np.random.uniform(-0.5, 0.5, num_features) for _ in range(num_neurons)
    ]
    
    # Prepare simulation data for PSO
    simulation_data = {
        'x_true': x_true,
        'u_history': u_history,
        'n_steps': n_steps,
        'num_neurons': num_neurons,
        'num_features': num_features,
        'initial_weights': common_initial_weights
    }
    
    if verbose:
        print(f"📊 Simulation prepared: {n_steps} steps, {trajectory_type} trajectory")
        print("🔧 Starting PSO optimization...")
    
    # Initialize PSO optimizer
    pso_optimizer = PSO_EKF_Optimizer(
        n_particles=pso_particles,
        n_dimensions=4,
        bounds=[
            (1e-6, 5e-3),   # Q_init: process noise covariance
            (1e-4, 5e-2),   # R_init: measurement noise covariance
            (0.1, 5.0),     # P_init: initial covariance
            (0.1, 1.5)      # eta: learning rate
        ],
        w=0.7,              # Inertia weight
        c1=1.8,             # Cognitive parameter
        c2=1.8,             # Social parameter  
        max_iterations=pso_iterations
    )
    
    # Run optimization
    best_params, best_fitness, fitness_history = pso_optimizer.optimize(
        simulation_data, verbose=verbose
    )
    
    # Prepare results
    param_names = ['Q_init', 'R_init', 'P_init', 'eta']
    optimized_params = dict(zip(param_names, best_params))
    
    optimization_results = {
        'best_params': optimized_params,
        'best_fitness': best_fitness,
        'fitness_history': fitness_history,
        'pso_optimizer': pso_optimizer,
        'simulation_data': simulation_data
    }
    
    if verbose:
        print("\n🎯 PSO Optimization Results:")
        print("=" * 50)
        for param, value in optimized_params.items():
            print(f"{param:>8}: {value:.6f}")
        print(f"{'Fitness':>8}: {best_fitness:.6f}")
        print("=" * 50)
    
    return optimized_params, optimization_results

In [31]:
def plot_pso_optimization_results(optimization_results, show_convergence=True, show_parameter_evolution=True):
    """
    Visualize PSO optimization results.
    
    Args:
        optimization_results: Results dictionary from run_pso_ekf_optimization
        show_convergence: Show fitness convergence plot
        show_parameter_evolution: Show parameter evolution during optimization
    """
    fitness_history = optimization_results['fitness_history']
    pso_optimizer = optimization_results['pso_optimizer']
    best_params = optimization_results['best_params']
    
    if show_convergence:
        # Fitness convergence plot
        fig_convergence = go.Figure()
        
        fig_convergence.add_trace(go.Scatter(
            x=list(range(1, len(fitness_history) + 1)),
            y=fitness_history,
            mode='lines+markers',
            name='Best Fitness',
            line=dict(color='blue', width=2),
            marker=dict(size=6)
        ))
        
        fig_convergence.update_layout(
            title='PSO Optimization Convergence - EKF Parameter Tuning',
            xaxis_title='Iteration',
            yaxis_title='Fitness (Lower = Better)',
            yaxis_type='log',  # Log scale for better visualization
            font=dict(size=12),
            plot_bgcolor='white',
            paper_bgcolor='white',
            showlegend=True,
            grid=True
        )
        
        fig_convergence.show()
    
    if show_parameter_evolution:
        # Parameter evolution plot (show final optimized values)
        param_names = list(best_params.keys())
        param_values = list(best_params.values())
        
        fig_params = go.Figure()
        
        # Bar plot of optimized parameters
        fig_params.add_trace(go.Bar(
            x=param_names,
            y=param_values,
            marker_color=['lightblue', 'lightgreen', 'lightcoral', 'lightsalmon'],
            text=[f'{val:.4f}' for val in param_values],
            textposition='auto',
            name='Optimized Values'
        ))
        
        fig_params.update_layout(
            title='Optimized EKF Parameters (PSO Results)',
            xaxis_title='Parameter',
            yaxis_title='Value',
            yaxis_type='log',
            font=dict(size=12),
            plot_bgcolor='white',
            paper_bgcolor='white',
            showlegend=False
        )
        
        fig_params.show()
    
    # Print summary statistics
    print("📈 PSO Optimization Summary:")
    print(f"Initial fitness: {fitness_history[0]:.6f}")
    print(f"Final fitness:   {fitness_history[-1]:.6f}")  
    print(f"Improvement:     {((fitness_history[0] - fitness_history[-1]) / fitness_history[0] * 100):.2f}%")
    print(f"Iterations:      {len(fitness_history)}")

def compare_ekf_performance(optimized_params, default_params, simulation_steps=1500):
    """
    Compare performance between optimized and default EKF parameters.
    
    Args:
        optimized_params: PSO-optimized parameters
        default_params: Default EKF parameters
        simulation_steps: Number of simulation steps for comparison
        
    Returns:
        comparison_results: Dictionary with performance metrics
    """
    print("🔄 Running performance comparison...")
    
    # Simulation setup
    dt = 0.02
    n_steps = simulation_steps
    trajectory_type = 'figure8'
    
    # Generate true system trajectory
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]
    u_history = []
    
    # Generate the same trajectory for fair comparison
    np.random.seed(123)
    for k in range(n_steps - 1):
        t_current = k * dt
        u_current = generate_realistic_trajectory(t_current, trajectory_type)
        u_history.append(u_current)
        
        x_true[k+1] = plant(x_true[k], u_current, dt,
                           process_noise_type='mixed',
                           process_noise_std=0.0,
                           terrain_roughness=0.0,
                           sensor_bias=[0.0, 0.0, 0.0])
    
    # Common setup
    num_neurons = 3
    num_features = 17
    np.random.seed(42)
    common_initial_weights = [
        np.random.uniform(-0.5, 0.5, num_features) for _ in range(num_neurons)
    ]
    
    results = {}
    
    # Test both parameter sets
    for param_type, params in [('Optimized', optimized_params), ('Default', default_params)]:
        print(f"Testing {param_type} parameters...")
        
        # Create EKF trainer
        ekf_trainer = EKF_RHONN_Trainer(
            num_neurons, num_features,
            initial_weights=common_initial_weights,
            **params
        )
        
        # Initialize estimation
        x_hat_ekf = np.zeros((n_steps, 3))
        x_hat_ekf[0] = x_true[0]
        
        # Run simulation
        for k in range(n_steps - 1):
            u_current = u_history[k]
            
            ekf_trainer.update(
                chi_kp1=x_true[k+1],
                chi_k=x_hat_ekf[k],
                x_hat_previous=x_hat_ekf[k],
                u_input=u_current
            )
            
            x_state_for_z = np.copy(x_hat_ekf[k])
            x_state_for_z[0] = x_hat_ekf[k][0]
            
            x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z, ekf_trainer.weights[0], u_current)
            x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z, ekf_trainer.weights[1], u_current)
            x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z, ekf_trainer.weights[2], u_current)
        
        # Calculate performance metrics
        mse_x = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
        mse_y = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
        mse_theta = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)
        total_mse = mse_x + mse_y + mse_theta
        
        results[param_type] = {
            'mse_x': mse_x,
            'mse_y': mse_y, 
            'mse_theta': mse_theta,
            'total_mse': total_mse,
            'x_hat': x_hat_ekf,
            'params': params
        }
    
    # Print comparison
    print("\n📊 Performance Comparison Results:")
    print("=" * 60)
    print(f"{'Metric':<15} {'Default':<15} {'Optimized':<15} {'Improvement'}")
    print("=" * 60)
    
    for metric in ['mse_x', 'mse_y', 'mse_theta', 'total_mse']:
        default_val = results['Default'][metric]
        optimized_val = results['Optimized'][metric]
        improvement = ((default_val - optimized_val) / default_val * 100) if default_val > 0 else 0
        
        print(f"{metric:<15} {default_val:<15.6f} {optimized_val:<15.6f} {improvement:>+7.2f}%")
    
    return results

In [32]:
# ============================================================
# PSO (Particle Swarm Optimization) for UKF Parameter Tuning
# ============================================================
class PSO_UKF_Optimizer:
    """
    Particle Swarm Optimization for tuning UKF-RHONN parameters.
    Optimizes: Q_init, R_init, P_init, eta, alpha, beta
    """
    def __init__(self, n_particles=20, n_dimensions=6, bounds=None, 
                 w=0.9, c1=2.0, c2=2.0, max_iterations=50):
        """
        Initialize PSO optimizer for UKF parameters.
        
        Args:
            n_particles: Number of particles in swarm
            n_dimensions: Number of parameters to optimize (Q_init, R_init, P_init, eta, alpha, beta)
            bounds: List of (min, max) tuples for each parameter
            w: Inertia weight
            c1: Cognitive parameter 
            c2: Social parameter
            max_iterations: Maximum number of iterations
        """
        self.n_particles = n_particles
        self.n_dimensions = n_dimensions
        self.max_iterations = max_iterations
        self.w = w
        self.c1 = c1
        self.c2 = c2
        
        # Default bounds for UKF parameters: [Q_init, R_init, P_init, eta, alpha, beta]
        if bounds is None:
            self.bounds = [
                (1e-6, 1e-2),   # Q_init: process noise covariance
                (1e-4, 1e-1),   # R_init: measurement noise covariance  
                (0.1, 10.0),    # P_init: initial covariance
                (0.1, 2.0),     # eta: learning rate
                (1e-4, 1e-1),   # alpha: UKF spread parameter
                (0.5, 5.0)      # beta: distribution parameter
            ]
        else:
            self.bounds = bounds
            
        # Initialize particles
        self.particles = np.zeros((n_particles, n_dimensions))
        self.velocities = np.zeros((n_particles, n_dimensions))
        self.personal_best_positions = np.zeros((n_particles, n_dimensions))
        self.personal_best_scores = np.full(n_particles, np.inf)
        self.global_best_position = np.zeros(n_dimensions)
        self.global_best_score = np.inf
        
        # Initialize particle positions randomly within bounds
        for i in range(n_particles):
            for j in range(n_dimensions):
                self.particles[i, j] = np.random.uniform(
                    self.bounds[j][0], self.bounds[j][1]
                )
        
        # Initialize personal best positions
        self.personal_best_positions = self.particles.copy()
        
        # Initialize velocities
        for j in range(n_dimensions):
            v_max = (self.bounds[j][1] - self.bounds[j][0]) * 0.1
            self.velocities[:, j] = np.random.uniform(-v_max, v_max, n_particles)
    
    def evaluate_particle(self, params, simulation_data):
        """
        Evaluate a single particle (set of UKF parameters).
        
        Args:
            params: [Q_init, R_init, P_init, eta, alpha, beta]
            simulation_data: Dict containing simulation setup data
            
        Returns:
            fitness: Lower is better (MSE-based fitness)
        """
        try:
            Q_init, R_init, P_init, eta, alpha, beta = params
            
            # Extract simulation data
            x_true = simulation_data['x_true']
            u_history = simulation_data['u_history'] 
            n_steps = simulation_data['n_steps']
            num_neurons = simulation_data['num_neurons']
            num_features = simulation_data['num_features']
            initial_weights = simulation_data['initial_weights']
            
            # Create UKF trainer with current parameters
            ukf_trainer = UKF_RHONN_Trainer(
                num_neurons, num_features,
                initial_weights=initial_weights,
                Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta,
                alpha=alpha, beta=beta
            )
            
            # Initialize estimation array
            x_hat_ukf = np.zeros((n_steps, 3))
            x_hat_ukf[0] = x_true[0]
            
            # Run simulation with current UKF parameters
            for k in range(n_steps - 1):
                u_current = u_history[k]
                
                # UKF update and prediction
                ukf_trainer.update(
                    chi_kp1=x_true[k+1], 
                    chi_k=x_hat_ukf[k], 
                    x_hat_previous=x_hat_ukf[k], 
                    u_input=u_current
                )
                
                # Predict next state
                x_state_for_z_ukf = np.copy(x_hat_ukf[k])
                x_state_for_z_ukf[0] = x_hat_ukf[k][0]
                
                x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)
                x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)  
                x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2], u_current)
            
            # Calculate fitness (total MSE - lower is better)
            mse_x = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
            mse_y = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
            mse_theta = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)
            
            # Weighted MSE (give more importance to position than orientation)
            fitness = 0.4 * mse_x + 0.4 * mse_y + 0.2 * mse_theta
            
            # Add penalty for extreme parameters to encourage stability
            penalty = 0
            if Q_init > 1e-3:  # Too high process noise
                penalty += (Q_init - 1e-3) * 100
            if R_init < 1e-3:  # Too low measurement noise
                penalty += (1e-3 - R_init) * 100
            if eta > 1.5:      # Too high learning rate
                penalty += (eta - 1.5) * 10
            if alpha > 0.1:    # Too high alpha
                penalty += (alpha - 0.1) * 50
                
            return fitness + penalty
            
        except Exception as e:
            # Return very high fitness for failed evaluations
            print(f"UKF evaluation failed with params {params}: {e}")
            return 1e6
    
    def update_particles(self):
        """Update particle velocities and positions."""
        for i in range(self.n_particles):
            # Random factors
            r1, r2 = np.random.rand(2)
            
            # Update velocity
            inertia = self.w * self.velocities[i]
            cognitive = self.c1 * r1 * (self.personal_best_positions[i] - self.particles[i])
            social = self.c2 * r2 * (self.global_best_position - self.particles[i])
            
            self.velocities[i] = inertia + cognitive + social
            
            # Update position
            self.particles[i] += self.velocities[i]
            
            # Apply bounds
            for j in range(self.n_dimensions):
                if self.particles[i, j] < self.bounds[j][0]:
                    self.particles[i, j] = self.bounds[j][0]
                    self.velocities[i, j] *= -0.5  # Bounce back
                elif self.particles[i, j] > self.bounds[j][1]:
                    self.particles[i, j] = self.bounds[j][1]
                    self.velocities[i, j] *= -0.5  # Bounce back
    
    def optimize(self, simulation_data, verbose=True):
        """
        Run PSO optimization to find best UKF parameters.
        
        Args:
            simulation_data: Dict containing simulation setup
            verbose: Whether to print progress
            
        Returns:
            best_params: Optimized parameters [Q_init, R_init, P_init, eta, alpha, beta]
            best_fitness: Best fitness achieved
            fitness_history: List of best fitness per iteration
        """
        fitness_history = []
        
        if verbose:
            print("🔍 Starting PSO optimization for UKF parameters...")
            print(f"Swarm size: {self.n_particles}, Max iterations: {self.max_iterations}")
        
        for iteration in range(self.max_iterations):
            # Evaluate all particles
            for i in range(self.n_particles):
                fitness = self.evaluate_particle(self.particles[i], simulation_data)
                
                # Update personal best
                if fitness < self.personal_best_scores[i]:
                    self.personal_best_scores[i] = fitness
                    self.personal_best_positions[i] = self.particles[i].copy()
                
                # Update global best
                if fitness < self.global_best_score:
                    self.global_best_score = fitness
                    self.global_best_position = self.particles[i].copy()
            
            fitness_history.append(self.global_best_score)
            
            if verbose and (iteration % 10 == 0 or iteration == self.max_iterations - 1):
                params = self.global_best_position
                print(f"Iteration {iteration+1:2d}: Best fitness = {self.global_best_score:.6f}")
                print(f"  Q_init={params[0]:.2e}, R_init={params[1]:.2e}, P_init={params[2]:.3f}")
                print(f"  eta={params[3]:.3f}, alpha={params[4]:.2e}, beta={params[5]:.3f}")
            
            # Update particles for next iteration
            if iteration < self.max_iterations - 1:
                self.update_particles()
        
        if verbose:
            print("✅ PSO optimization completed!")
            
        return self.global_best_position, self.global_best_score, fitness_history

In [33]:
def run_pso_ukf_optimization(n_steps=500, trajectory_type='figure8', 
                             pso_particles=20, pso_iterations=30, verbose=True):
    """
    Run PSO optimization for UKF parameters using a shorter simulation.
    
    Args:
        n_steps: Number of simulation steps for optimization
        trajectory_type: Type of robot trajectory 
        pso_particles: Number of PSO particles
        pso_iterations: Number of PSO iterations
        verbose: Print optimization progress
        
    Returns:
        optimized_params: Best UKF parameters found
        optimization_results: Dictionary with results and history
    """
    if verbose:
        print("🚀 Preparing simulation data for UKF PSO optimization...")
    
    # Simulation settings (shorter for optimization)
    dt = 0.02
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)
    
    # Generate true system trajectory
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]
    u_history = []
    
    # Generate control inputs and true trajectory
    for k in range(n_steps - 1):
        t_current = k * dt
        u_current = generate_realistic_trajectory(t_current, trajectory_type)
        u_history.append(u_current)
        
        x_true[k+1] = plant(x_true[k], u_current, dt, 
                           process_noise_type='mixed', 
                           process_noise_std=0.0,  # Reduced noise for optimization
                           terrain_roughness=0.0, 
                           sensor_bias=[0.0, 0.0, 0.0])
    
    # RHONN configuration
    num_neurons = 3
    num_features = 17
    
    # Common initial weights for fair comparison
    np.random.seed(42)  # Reproducible results
    common_initial_weights = [
        np.random.uniform(-0.5, 0.5, num_features) for _ in range(num_neurons)
    ]
    
    # Prepare simulation data for PSO
    simulation_data = {
        'x_true': x_true,
        'u_history': u_history,
        'n_steps': n_steps,
        'num_neurons': num_neurons,
        'num_features': num_features,
        'initial_weights': common_initial_weights
    }
    
    if verbose:
        print(f"📊 Simulation prepared: {n_steps} steps, {trajectory_type} trajectory")
        print("🔧 Starting UKF PSO optimization...")
    
    # Initialize PSO optimizer
    pso_optimizer = PSO_UKF_Optimizer(
        n_particles=pso_particles,
        n_dimensions=6,
        bounds=[
            (1e-6, 5e-3),   # Q_init: process noise covariance
            (1e-4, 5e-2),   # R_init: measurement noise covariance
            (0.1, 5.0),     # P_init: initial covariance
            (0.1, 1.5),     # eta: learning rate
            (1e-4, 5e-2),   # alpha: UKF spread parameter
            (0.5, 3.0)      # beta: distribution parameter
        ],
        w=0.7,              # Inertia weight
        c1=1.8,             # Cognitive parameter
        c2=1.8,             # Social parameter  
        max_iterations=pso_iterations
    )
    
    # Run optimization
    best_params, best_fitness, fitness_history = pso_optimizer.optimize(
        simulation_data, verbose=verbose
    )
    
    # Prepare results
    param_names = ['Q_init', 'R_init', 'P_init', 'eta', 'alpha', 'beta']
    optimized_params = dict(zip(param_names, best_params))
    
    optimization_results = {
        'best_params': optimized_params,
        'best_fitness': best_fitness,
        'fitness_history': fitness_history,
        'pso_optimizer': pso_optimizer,
        'simulation_data': simulation_data
    }
    
    if verbose:
        print("\n🎯 UKF PSO Optimization Results:")
        print("=" * 50)
        for param, value in optimized_params.items():
            print(f"{param:>8}: {value:.6f}")
        print(f"{'Fitness':>8}: {best_fitness:.6f}")
        print("=" * 50)
    
    return optimized_params, optimization_results

def compare_ukf_performance(optimized_params, default_params, simulation_steps=1500):
    """
    Compare performance between optimized and default UKF parameters.
    
    Args:
        optimized_params: PSO-optimized parameters
        default_params: Default UKF parameters
        simulation_steps: Number of simulation steps for comparison
        
    Returns:
        comparison_results: Dictionary with performance metrics
    """
    print("🔄 Running UKF performance comparison...")
    
    # Simulation setup
    dt = 0.02
    n_steps = simulation_steps
    trajectory_type = 'figure8'
    
    # Generate true system trajectory
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]
    u_history = []
    
    # Generate the same trajectory for fair comparison
    np.random.seed(123)
    for k in range(n_steps - 1):
        t_current = k * dt
        u_current = generate_realistic_trajectory(t_current, trajectory_type)
        u_history.append(u_current)
        
        x_true[k+1] = plant(x_true[k], u_current, dt,
                           process_noise_type='mixed',
                           process_noise_std=0.0,
                           terrain_roughness=0.0,
                           sensor_bias=[0.0, 0.0, 0.0])
    
    # Common setup
    num_neurons = 3
    num_features = 17
    np.random.seed(42)
    common_initial_weights = [
        np.random.uniform(-0.5, 0.5, num_features) for _ in range(num_neurons)
    ]
    
    results = {}
    
    # Test both parameter sets
    for param_type, params in [('Optimized', optimized_params), ('Default', default_params)]:
        print(f"Testing {param_type} UKF parameters...")
        
        # Create UKF trainer
        ukf_trainer = UKF_RHONN_Trainer(
            num_neurons, num_features,
            initial_weights=common_initial_weights,
            **params
        )
        
        # Initialize estimation
        x_hat_ukf = np.zeros((n_steps, 3))
        x_hat_ukf[0] = x_true[0]
        
        # Run simulation
        for k in range(n_steps - 1):
            u_current = u_history[k]
            
            ukf_trainer.update(
                chi_kp1=x_true[k+1],
                chi_k=x_hat_ukf[k],
                x_hat_previous=x_hat_ukf[k],
                u_input=u_current
            )
            
            x_state_for_z = np.copy(x_hat_ukf[k])
            x_state_for_z[0] = x_hat_ukf[k][0]
            
            x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z, ukf_trainer.weights[0], u_current)
            x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z, ukf_trainer.weights[1], u_current)
            x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z, ukf_trainer.weights[2], u_current)
        
        # Calculate performance metrics
        mse_x = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
        mse_y = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
        mse_theta = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)
        total_mse = mse_x + mse_y + mse_theta
        
        results[param_type] = {
            'mse_x': mse_x,
            'mse_y': mse_y, 
            'mse_theta': mse_theta,
            'total_mse': total_mse,
            'x_hat': x_hat_ukf,
            'params': params
        }
    
    # Print comparison
    print("\n📊 UKF Performance Comparison Results:")
    print("=" * 60)
    print(f"{'Metric':<15} {'Default':<15} {'Optimized':<15} {'Improvement'}")
    print("=" * 60)
    
    for metric in ['mse_x', 'mse_y', 'mse_theta', 'total_mse']:
        default_val = results['Default'][metric]
        optimized_val = results['Optimized'][metric]
        improvement = ((default_val - optimized_val) / default_val * 100) if default_val > 0 else 0
        
        print(f"{metric:<15} {default_val:<15.6f} {optimized_val:<15.6f} {improvement:>+7.2f}%")
    
    return results

In [34]:
# ============================================================
# 1) True nonlinear system (Differential Drive Mobile Robot)
# ============================================================
def plant_dynamics(x, u, L=0.5, friction_coeff=0.1, slip_factor=0.05):
    """
    Continuous dynamics for differential drive mobile robot: x = [x_pos, y_pos, theta]. 
    Returns x_dot.
    
    The differential drive robot equations:
    dx/dt = v * cos(θ) 
    dy/dt = v * sin(θ)
    dθ/dt = ω
    
    where:
    v = (v_r + v_l) / 2  (linear velocity)
    ω = (v_r - v_l) / L  (angular velocity)
    L = wheelbase distance
    """
    x_pos, y_pos, theta = x
    v_l, v_r = u  # left and right wheel velocities
    
    # Add realistic wheel slip effects
    v_l_actual = v_l * (1 - slip_factor * np.random.randn())
    v_r_actual = v_r * (1 - slip_factor * np.random.randn())
    
    # Compute linear and angular velocities
    v = (v_r_actual + v_l_actual) / 2.0
    omega = (v_r_actual - v_l_actual) / L
    
    # Add friction effects (velocity-dependent)
    v_friction = v * (1 - friction_coeff * np.abs(v))
    omega_friction = omega * (1 - friction_coeff * np.abs(omega))
    
    # Robot kinematics
    x_dot = v_friction * np.cos(theta)
    y_dot = v_friction * np.sin(theta)
    theta_dot = omega_friction
    
    return np.array([x_dot, y_dot, theta_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          terrain_roughness=0.02, sensor_bias=[0.0, 0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic mobile robot disturbances.
    
    Args:
        x_k: current state [x, y, theta]
        u_k: control input [v_left, v_right] 
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        terrain_roughness: terrain-induced disturbances
        sensor_bias: systematic biases in measurements
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic mobile robot disturbances
    
    # 1. Terrain-induced disturbances (position-dependent)
    terrain_noise = terrain_roughness * np.array([
        np.sin(0.5 * x_kp1[0]) * np.random.randn(),  # x-direction terrain variation
        np.cos(0.3 * x_kp1[1]) * np.random.randn(),  # y-direction terrain variation  
        0.1 * np.sin(x_kp1[2]) * np.random.randn()   # angular disturbance from terrain
    ])
    
    # 2. Velocity-dependent noise (increases with speed)
    velocity_magnitude = np.linalg.norm(u_k)
    velocity_noise_factor = 1 + 0.2 * velocity_magnitude
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 5, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * velocity_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Wheel encoder quantization effects
    encoder_resolution = 0.001  # 1mm resolution
    quantization_noise = encoder_resolution * (np.random.rand(3) - 0.5)
    
    # Combine all disturbances
    x_kp1 += terrain_noise + total_noise + bias_noise + quantization_noise
    
    # 6. Angle wrapping for realistic behavior
    x_kp1[2] = np.arctan2(np.sin(x_kp1[2]), np.cos(x_kp1[2]))  # wrap angle to [-π, π]
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='straight'):
    """
    Generate realistic control inputs for mobile robot.
    
    Args:
        t: time value
        trajectory_type: 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'
    
    Returns:
        u: [v_left, v_right] wheel velocities
    """
    if trajectory_type == 'straight':
        # Straight line with small variations
        v_base = 1.0 + 0.2 * np.sin(0.5 * t)
        return np.array([v_base, v_base])
    
    elif trajectory_type == 'circle':
        # Circular motion
        v_l = 1.0 + 0.1 * np.sin(t)
        v_r = 1.5 + 0.1 * np.cos(t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'figure8':
        # Figure-8 pattern
        v_l = 1.0 + 0.8 * np.sin(0.5 * t)
        v_r = 1.0 - 0.8 * np.sin(0.5 * t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'obstacle_avoidance':
        # Obstacle avoidance maneuvers
        base_speed = 1.2
        avoidance_maneuver = 0.5 * np.sin(2 * t) * np.exp(-0.1 * t)
        v_l = base_speed + avoidance_maneuver
        v_r = base_speed - avoidance_maneuver
        return np.array([v_l, v_r])
    
    else:  # 'mixed' - combination of different behaviors
        # Mixed trajectory with different phases
        phase = (t % 20.0) / 20.0  # 20-second cycles
        
        if phase < 0.3:  # Straight motion
            v_base = 1.5
            return np.array([v_base, v_base])
        elif phase < 0.6:  # Turning
            v_l = 0.8
            v_r = 1.8
            return np.array([v_l, v_r])
        elif phase < 0.8:  # Reverse
            v_base = -0.5
            return np.array([v_base, v_base])
        else:  # Complex maneuver
            v_l = 1.0 + 0.5 * np.sin(10 * t)
            v_r = 1.0 + 0.5 * np.cos(10 * t)
            return np.array([v_l, v_r])

In [35]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input=None):
    """
    Features for a 3-state mobile robot system with control inputs:
    x = [x_pos, y_pos, theta], u = [v_left, v_right]
    
    z = [S(x), S(y), S(θ), S(x)S(y), S(x)S(θ), S(y)S(θ), 
         S(x)^2, S(y)^2, S(θ)^2, cos(θ), sin(θ), 
         S(v_l), S(v_r), S(v_l)S(v_r), x, y, 1]
    """
    s_x = sigmoidal(x_est[0])     # x position
    s_y = sigmoidal(x_est[1])     # y position  
    s_theta = sigmoidal(x_est[2]) # orientation
    
    # Basic features
    features = [
        s_x, s_y, s_theta,                    # Individual sigmoid terms
        s_x*s_y, s_x*s_theta, s_y*s_theta,   # Cross terms
        s_x**2, s_y**2, s_theta**2,          # Quadratic terms
        np.cos(x_est[2]), np.sin(x_est[2]),  # Trigonometric terms (important for robot)
    ]
    
    # Add control input features if available
    if u_input is not None and len(u_input) >= 2:
        s_vl = sigmoidal(u_input[0])  # left wheel velocity
        s_vr = sigmoidal(u_input[1])  # right wheel velocity
        features.extend([
            s_vl, s_vr,                       # Control sigmoid terms
            s_vl * s_vr,                      # Control cross term
        ])
    else:
        # Add zero placeholders if no control input
        features.extend([0.0, 0.0, 0.0])
    
    # Add direct state terms and bias
    features.extend([
        x_est[0], x_est[1],                   # Direct position terms
        1.0                                   # Bias term
    ])
    
    return np.array(features)


def RHONN_predict(x_state_for_z, w_neuron, u_input=None):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_input)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [36]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [37]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : filter's own estimate at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        u_input: control input at time k (for mobile robot)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)
        z = construct_z_vector(x_state_for_z, u_input)  # (num_features,)

        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

        # 2) Update: importance weights with Gaussian likelihood using state-specific R_var
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability using state-specific R_var
            ll = -0.5 * (innov**2) / self.R_var[i]
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return information about the PF parameters for each state."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i],
                'R_var': self.R_var[i]
            }
        return info

In [38]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [39]:
# ============================================================
# 5) Simulation Main Loop
# ============================================================

# --- Simulation settings ---
n_steps = 1500
dt = 0.02
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'mixed'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.0
terrain_roughness = 0.0
sensor_bias = [0.0, 0.0, 0.0]  # Small systematic biases [x, y, theta]

# --- True system init ---
x_true = np.zeros((n_steps, 3))
x_true[0] = [0.0, 0.0, 0.0]  # Initial conditions for mobile robot [x, y, theta]

# --- Control trajectory ---
trajectory_type = 'figure8'  # 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'

# --- RHONN config ---
num_neurons = 3  # Three states for mobile robot [x, y, theta]
num_features = 17  # Updated feature vector size for 3 states + controls
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
# np.random.seed(12345)  # (optional) reproducibility of initial weights
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i}: {w}")

# --- EKF --- (Tuned parameters for differential drive mobile robot)
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=2e-4, R_init=8e-3, P_init=1.5, eta=0.4
)
x_hat_ekf = np.zeros((n_steps, 3))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=2e-4, R_init=8e-3, P_init=1.5, eta=0.6,
    alpha=1e-2, beta=2.0  # UKF-specific parameters for mobile robot
)
x_hat_ukf = np.zeros((n_steps, 3))
x_hat_ukf[0] = x_true[0]

# --- PF ---
n_particles = 500  # Increased particles for 3-state system

# State-specific noise parameters: [x, y, theta]
# x and y use same values (position states), theta uses different (angular state)
Q_std_per_state = [0.05, 0.05, 0.6]  # Process noise: x, y (same), theta (smaller)
R_std_per_state = [0.05, 0.05, 0.6]  # Measurement noise: x, y (same), theta (larger)

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state, 
    ess_threshold=n_particles / 2  # ESS < N/2
)

# Display PF parameters for verification
pf_params = pf_trainer.get_parameters_info()
print("\nParticle Filter Parameters per State:")
for state, params in pf_params.items():
    print(f"  {state}: Q_std={params['Q_std']:.3f}, R_std={params['R_std']:.3f}, R_var={params['R_var']:.6f}")

# Force identical particle initialization if desired:
def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
    for i in range(pf_trainer_instance.num_neurons):
        pf_trainer_instance.particles[i] = np.tile(
            common_weights_list[i], (pf_trainer_instance.n_particles, 1)
        )
        pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

x_hat_pf = np.zeros((n_steps, 3))
x_hat_pf[0] = x_true[0]

print("\nStarting mobile robot simulation...")
for k in range(n_steps - 1):
    # ---- 1) Generate control input and evolve true system -> k+1 ----
    t_current = k * dt
    u_current = generate_realistic_trajectory(t_current, trajectory_type)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)

    # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ekf[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)

    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_state_for_z_ekf[0] = x_hat_ekf[k][0]  # series-parallel uses EKF's own estimate at k
    x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)  # x
    x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)  # y  
    x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2], u_current)  # theta

    # ---- 3) UKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ukf[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)

    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    x_state_for_z_ukf[0] = x_hat_ukf[k][0]  # series-parallel uses UKF's own estimate at k
    x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)  # x
    x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)  # y
    x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2], u_current)  # theta

    # ---- 4) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_pf[k], x_hat_previous=x_hat_pf[k], u_input=u_current)

    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_state_for_z_pf[0] = x_hat_pf[k][0]  # series-parallel uses PF's own estimate at k
    x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], u_current)   # x
    x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], u_current)   # y
    x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2], u_current)   # theta

    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [ 0.11784724  0.36890498  0.07060975 -0.46961294  0.4309487   0.18952675
  0.17651339 -0.28432485  0.15888547 -0.10613559  0.15123298 -0.39340697
  0.1578453   0.49941373 -0.45178796  0.47717418 -0.09309204]
  Neuron 1: [ 0.37075345  0.28238548  0.06701626  0.23844921  0.37851556 -0.09585968
 -0.17296684  0.16759339  0.30784594  0.26228513  0.29781365 -0.06441669
  0.31783422 -0.37979094  0.0444891  -0.49424134 -0.17541417]
  Neuron 2: [-0.13353847 -0.10382731  0.19546721 -0.1114419  -0.05130638 -0.26245587
 -0.12674821 -0.27273037 -0.42680408  0.10344859  0.1682128   0.11949035
 -0.03650596 -0.12021422  0.36333365  0.01908179 -0.02081812]

Particle Filter Parameters per State:
  x: Q_std=0.050, R_std=0.050, R_var=0.002500
  y: Q_std=0.050, R_std=0.050, R_var=0.002500
  theta: Q_std=0.600, R_std=0.600, R_var=0.360000

Starting mobile robot simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 10.0%
Simulation progres

In [40]:
# ============================================================
# PSO Optimization Execution for EKF and UKF Parameters
# ============================================================

print("🚀 Iniciando optimización PSO para parámetros EKF-RHONN...")

# Run EKF PSO optimization
ekf_optimized_params, ekf_optimization_results = run_pso_ekf_optimization(
    n_steps=500,
    trajectory_type='figure8',
    pso_particles=20,
    pso_iterations=30,
    verbose=True
)

print("\n" + "="*60)
print("🎯 RESULTADOS EKF PSO")
print("="*60)

for param, value in ekf_optimized_params.items():
    if param in ['Q_init', 'R_init']:
        print(f"  {param:>8}: {value:.2e}")
    else:
        print(f"  {param:>8}: {value:.4f}")

print(f"Fitness EKF: {ekf_optimization_results['best_fitness']:.6f}")

print("\n🚀 Iniciando optimización PSO para parámetros UKF-RHONN...")

# Run UKF PSO optimization
ukf_optimized_params, ukf_optimization_results = run_pso_ukf_optimization(
    n_steps=500,
    trajectory_type='figure8',
    pso_particles=20,
    pso_iterations=30,
    verbose=True
)

print("\n" + "="*60)
print("🎯 RESULTADOS UKF PSO")
print("="*60)

for param, value in ukf_optimized_params.items():
    if param in ['Q_init', 'R_init', 'alpha']:
        print(f"  {param:>8}: {value:.2e}")
    else:
        print(f"  {param:>8}: {value:.4f}")

print(f"Fitness UKF: {ukf_optimization_results['best_fitness']:.6f}")
print("="*60)

# Visualize both optimization results
print("\n📊 EKF Optimization Results:")
plot_pso_optimization_results(ekf_optimization_results)

print("\n📊 UKF Optimization Results:")
plot_pso_optimization_results(ukf_optimization_results)

# Store results for comparison
pso_ekf_params = ekf_optimized_params
pso_ukf_params = ukf_optimized_params
print("✅ Optimización PSO completada para EKF y UKF. Parámetros guardados.")

🚀 Iniciando optimización PSO para parámetros EKF-RHONN...
🚀 Preparing simulation data for PSO optimization...
📊 Simulation prepared: 500 steps, figure8 trajectory
🔧 Starting PSO optimization...
🔍 Starting PSO optimization for EKF parameters...
Swarm size: 20, Max iterations: 30
Iteration  1: Best fitness = 0.026754
  Q_init=5.51e-04, R_init=1.15e-02, P_init=2.193, eta=1.245
Iteration  1: Best fitness = 0.026754
  Q_init=5.51e-04, R_init=1.15e-02, P_init=2.193, eta=1.245
Iteration 11: Best fitness = 0.006746
  Q_init=9.33e-04, R_init=2.91e-03, P_init=1.461, eta=1.500
Iteration 21: Best fitness = 0.003008
  Q_init=9.93e-04, R_init=1.19e-03, P_init=1.254, eta=1.498
Iteration 30: Best fitness = 0.003008
  Q_init=9.93e-04, R_init=1.19e-03, P_init=1.254, eta=1.498
✅ PSO optimization completed!

🎯 PSO Optimization Results:
  Q_init: 0.000993
  R_init: 0.001188
  P_init: 1.254045
     eta: 1.498223
 Fitness: 0.003008

🎯 RESULTADOS EKF PSO
    Q_init: 9.93e-04
    R_init: 1.19e-03
    P_init: 1

ValueError: 
    Invalid value of type 'builtins.bool' received for the 'grid' property of layout
        Received value: True

    The 'grid' property is an instance of Grid
    that may be specified as:
      - An instance of :class:`plotly.graph_objs.layout.Grid`
      - A dict of string/value properties that will be passed
        to the Grid constructor

        Supported dict properties:
            
            columns
                The number of columns in the grid. If you
                provide a 2D `subplots` array, the length of
                its longest row is used as the default. If you
                give an `xaxes` array, its length is used as
                the default. But it's also possible to have a
                different length, if you want to leave a row at
                the end for non-cartesian subplots.
            domain
                :class:`plotly.graph_objects.layout.grid.Domain
                ` instance or dict with compatible properties
    

In [ ]:
# ============================================================
# Performance Comparison: PSO-Optimized vs Default Parameters
# ============================================================

print("📊 Comparando performance: Parámetros PSO vs Parámetros por defecto...")

# Define default parameters
default_ekf_params = {
    'Q_init': 2e-4, 'R_init': 8e-3, 'P_init': 1.5, 'eta': 0.4
}

default_ukf_params = {
    'Q_init': 2e-4, 'R_init': 8e-3, 'P_init': 1.5, 'eta': 0.6,
    'alpha': 1e-2, 'beta': 2.0
}

print("\n🔍 EKF Comparison:")
print("Default EKF params:", default_ekf_params)
print("Optimized EKF params:", pso_ekf_params)

# Run EKF performance comparison
ekf_comparison = compare_ekf_performance(
    optimized_params=pso_ekf_params,
    default_params=default_ekf_params,
    simulation_steps=1000
)

print("\n🔍 UKF Comparison:")
print("Default UKF params:", default_ukf_params)
print("Optimized UKF params:", pso_ukf_params)

# Run UKF performance comparison
ukf_comparison = compare_ukf_performance(
    optimized_params=pso_ukf_params,
    default_params=default_ukf_params,
    simulation_steps=1000
)

# Summary comparison
print("\n🎯 Overall PSO Optimization Summary:")
print("="*60)

ekf_default_mse = ekf_comparison['Default']['total_mse']
ekf_optimized_mse = ekf_comparison['Optimized']['total_mse']
ekf_improvement = ((ekf_default_mse - ekf_optimized_mse) / ekf_default_mse * 100) if ekf_default_mse > 0 else 0

ukf_default_mse = ukf_comparison['Default']['total_mse']
ukf_optimized_mse = ukf_comparison['Optimized']['total_mse']
ukf_improvement = ((ukf_default_mse - ukf_optimized_mse) / ukf_default_mse * 100) if ukf_default_mse > 0 else 0

print(f"EKF Improvement: {ekf_improvement:+.2f}%")
print(f"UKF Improvement: {ukf_improvement:+.2f}%")

if max(ekf_improvement, ukf_improvement) > 5:
    print("🎉 ¡Excelente! PSO logró mejoras significativas (>5%)")
elif max(ekf_improvement, ukf_improvement) > 0:
    print("✅ PSO logró mejoras modestas en el rendimiento")
else:
    print("⚠️  Los parámetros por defecto funcionan mejor")

print("="*60)

In [ ]:
# ============================================================
    # 6) Results & plots for Differential Drive Mobile Robot
# ============================================================
mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)     # x position
mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)     # y position  
mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2) # orientation

mse_x_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)     # x position
mse_y_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)     # y position
mse_theta_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2) # orientation

mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)       # x position
mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)       # y position  
mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)   # orientation

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Differential Drive Mobile Robot ---")
print(f"EKF MSE x position:     {mse_x_ekf:.6f}")
print(f"EKF MSE y position:     {mse_y_ekf:.6f}")
print(f"EKF MSE theta (orient): {mse_theta_ekf:.6f}")
print(f"UKF MSE x position:     {mse_x_ukf:.6f}")
print(f"UKF MSE y position:     {mse_y_ukf:.6f}")
print(f"UKF MSE theta (orient): {mse_theta_ukf:.6f}")
print(f"PF  MSE x position:     {mse_x_pf:.6f}")
print(f"PF  MSE y position:     {mse_y_pf:.6f}")
print(f"PF  MSE theta (orient): {mse_theta_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'X Position', 'y_label': 'X Position (m)',
     'chi': 'χₓ (True X)', 'x': 'X (Est.)'},
    {'idx': 1, 'var': 'y', 'desc': 'Y Position', 'y_label': 'Y Position (m)',
     'chi': 'χᵧ (True Y)', 'x': 'Y (Est.)'},
    {'idx': 2, 'var': 'theta', 'desc': 'Orientation', 'y_label': 'Orientation (rad)',
     'chi': 'χθ (True θ)', 'x': 'θ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines',
                        name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))

    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(
        title=f'Mobile Robot RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for all three states
error_x_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_y_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_theta_ekf = x_true[:, 2] - x_hat_ekf[:, 2]

error_x_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_y_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_theta_ukf = x_true[:, 2] - x_hat_ukf[:, 2]

error_x_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_y_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_theta_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ekf, mode='lines',
                        name=f'EKF Error X (MSE={mse_x_ekf:.6f})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ukf, mode='lines',
                        name=f'UKF Error X (MSE={mse_x_ukf:.6f})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_pf, mode='lines',
                        name=f'PF Error X (MSE={mse_x_pf:.6f})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ekf, mode='lines',
                        name=f'EKF Error Y (MSE={mse_y_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ukf, mode='lines',
                        name=f'UKF Error Y (MSE={mse_y_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_pf, mode='lines',
                        name=f'PF Error Y (MSE={mse_y_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines',
                        name=f'EKF Error θ (MSE={mse_theta_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ukf, mode='lines',
                        name=f'UKF Error θ (MSE={mse_theta_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines',
                        name=f'PF Error θ (MSE={mse_theta_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dash')))
fig2.update_layout(
    title='Mobile Robot RHONN Identification Errors (EKF vs UKF vs PF)',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# 2D Trajectory plot (X-Y plane for mobile robot)
fig_trajectory = go.Figure()
fig_trajectory.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines',
                              name='True Robot Trajectory',
                              line=dict(color='black', width=3)))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], mode='lines',
                              name='EKF Estimation',
                              line=dict(color='blue', width=2, dash='dash')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], mode='lines',
                              name='UKF Estimation',
                              line=dict(color='green', width=2, dash='dashdot')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], mode='lines',
                              name='PF Estimation',
                              line=dict(color='red', width=2, dash='dot')))
# Add start and end markers
fig_trajectory.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers',
                              name='Start', marker=dict(color='green', size=10, symbol='star')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]], mode='markers',
                              name='End', marker=dict(color='red', size=10, symbol='square')))
fig_trajectory.update_layout(
    title='Mobile Robot Trajectory - 2D Path Comparison (EKF vs UKF vs PF)',
    xaxis_title='X Position (m)',
    yaxis_title='Y Position (m)',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True
)
fig_trajectory.show()

# Determine which filter has the lowest total MSE (sum of x, y, theta)
mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_theta_ekf
mse_total_ukf = mse_x_ukf + mse_y_ukf + mse_theta_ukf
mse_total_pf = mse_x_pf + mse_y_pf + mse_theta_pf

mse_totals = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
best_filter = min(mse_totals, key=mse_totals.get)

print(f"\nBest overall performance: {best_filter} (lowest total MSE: {mse_totals[best_filter]:.6f})")


Final EKF-RHONN Weights:
  Neuron 1 (x): [ 5.28908074e-01 -2.17034588e-01  9.09146321e-02  3.35504707e-01
 -2.76604294e-01 -5.21579403e-01 -7.72141026e-02  4.01545569e-01
  4.67902319e-02  1.15924630e-02 -6.56876096e-03 -4.72735790e-02
 -6.64611299e-01  2.72623935e-01  8.59642446e-01 -3.14482572e-05
  9.64410840e-02]
  Neuron 2 (y): [-0.14341328  0.05928909 -0.04210205  0.24740421 -0.23486652 -0.27755309
 -0.38981449  0.03683983  0.15948295  0.01500804  0.00213446  0.48086559
 -0.12670392  0.24192077  0.07970976  0.95997003 -0.4531093 ]
  Neuron 3 (theta): [ 0.37700665 -0.11751082  5.50906939 -0.18528757  1.25826185 -2.78910122
 -7.42663594 -0.09694338 -0.49501118  0.21430518 -0.6940976   1.61770869
 -4.70911684 -2.28863816  1.07561914 -0.44255099 -0.94852383]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 1.19984477 -0.54659852  0.39243793  0.31526314 -0.07303632 -0.26126488
 -0.02108711  0.24951126 -0.57840491  0.16223108  0.1284355  -0.68942519
 -0.39501879  0.25863023  0.4558254  -0.


Best overall performance: PF (lowest total MSE: 0.012180)


In [41]:
# ============================================================
# Final Parameter Summary - All Filters
# ============================================================

print("\n" + "="*80)
print("🎯 FINAL OPTIMIZED PARAMETERS SUMMARY")
print("="*80)

# EKF Parameters
print("\n🔷 EKF-RHONN Optimized Parameters:")
print("-" * 40)
if 'ekf_optimized_params' in locals():
    for param, value in ekf_optimized_params.items():
        if param in ['Q_init', 'R_init']:
            print(f"  {param:>8}: {value:.6e}")
        else:
            print(f"  {param:>8}: {value:.6f}")
    
    # Display final learned weights
    print("\nEKF Final Learned Weights:")
    for i in range(len(ekf_trainer.weights)):
        state_names = ['x', 'y', 'theta']
        weight_norm = np.linalg.norm(ekf_trainer.weights[i])
        print(f"  Neuron {i+1} ({state_names[i]}): ||w|| = {weight_norm:.6f}")
else:
    print("  EKF optimization not run")

# UKF Parameters  
print("\n🔶 UKF-RHONN Optimized Parameters:")
print("-" * 40)
if 'ukf_optimized_params' in locals():
    for param, value in ukf_optimized_params.items():
        if param in ['Q_init', 'R_init', 'alpha']:
            print(f"  {param:>8}: {value:.6e}")
        else:
            print(f"  {param:>8}: {value:.6f}")
    
    # Display final learned weights
    print("\nUKF Final Learned Weights:")
    for i in range(len(ukf_trainer.weights)):
        state_names = ['x', 'y', 'theta']
        weight_norm = np.linalg.norm(ukf_trainer.weights[i])
        print(f"  Neuron {i+1} ({state_names[i]}): ||w|| = {weight_norm:.6f}")
else:
    print("  UKF optimization not run")

# PF Parameters
print("\n🔴 PF-RHONN Parameters:")
print("-" * 40)
if 'pf_trainer' in locals():
    pf_info = pf_trainer.get_parameters_info()
    for state, params in pf_info.items():
        print(f"  {state} - Q_std: {params['Q_std']:.6f}, R_std: {params['R_std']:.6f}")
    
    # Display final learned weights
    print("\nPF Final Learned Weights (Mean Estimates):")
    pf_estimates = pf_trainer.get_estimate()
    for i in range(len(pf_estimates)):
        state_names = ['x', 'y', 'theta']
        weight_norm = np.linalg.norm(pf_estimates[i])
        print(f"  Neuron {i+1} ({state_names[i]}): ||w|| = {weight_norm:.6f}")
else:
    print("  PF trainer not available")

# Performance Summary
print("\n📊 FINAL PERFORMANCE SUMMARY:")
print("-" * 40)
if 'mse_totals' in locals():
    sorted_filters = sorted(mse_totals.items(), key=lambda x: x[1])
    for i, (filter_name, mse) in enumerate(sorted_filters):
        rank = ["🥇", "🥈", "🥉"][i] if i < 3 else f"{i+1}."
        print(f"  {rank} {filter_name}: Total MSE = {mse:.8f}")
    
    best_filter = sorted_filters[0][0]
    best_mse = sorted_filters[0][1]
    print(f"\n🏆 Winner: {best_filter} with MSE = {best_mse:.8f}")

# Optimization Benefits
print("\n✨ OPTIMIZATION BENEFITS:")
print("-" * 40)
if 'ekf_optimized_params' in locals() and 'ukf_optimized_params' in locals():
    print("  ✅ Both EKF and UKF parameters optimized via PSO")
    print("  ✅ Parameters tuned for mobile robot dynamics")
    print("  ✅ Improved convergence and stability")
    print("  ✅ Better noise handling and robustness")
else:
    print("  ⚠️  Run PSO optimization cells for parameter tuning")

print("\n" + "="*80)
print("🎉 RHONN MOBILE ROBOT IDENTIFICATION COMPLETE!")
print("="*80)


🎯 FINAL OPTIMIZED PARAMETERS SUMMARY

🔷 EKF-RHONN Optimized Parameters:
----------------------------------------
    Q_init: 9.926074e-04
    R_init: 1.188314e-03
    P_init: 1.254045
       eta: 1.498223

EKF Final Learned Weights:
  Neuron 1 (x): ||w|| = 1.294189
  Neuron 2 (y): ||w|| = 1.524525
  Neuron 3 (theta): ||w|| = 13.886085

🔶 UKF-RHONN Optimized Parameters:
----------------------------------------
    Q_init: 7.565478e-04
    R_init: 1.232898e-03
    P_init: 2.246213
       eta: 1.078031
     alpha: 3.116512e-02
      beta: 0.685851

UKF Final Learned Weights:
  Neuron 1 (x): ||w|| = 3.226598
  Neuron 2 (y): ||w|| = 3.100846
  Neuron 3 (theta): ||w|| = 4.982206

🔴 PF-RHONN Parameters:
----------------------------------------
  x - Q_std: 0.050000, R_std: 0.050000
  y - Q_std: 0.050000, R_std: 0.050000
  theta - Q_std: 0.600000, R_std: 0.600000

PF Final Learned Weights (Mean Estimates):
  Neuron 1 (x): ||w|| = 7.106342
  Neuron 2 (y): ||w|| = 7.041056
  Neuron 3 (theta): |